# ch05 练习：分类模型

与讲解版逐 Cell 对应。核心代码被挖空，`assert` 验收**保留**——
跑通所有 `assert` 即自查通过。卡住就回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 模型训练 40% | 逻辑回归 / KNN / 朴素贝叶斯 / SVM / DT / RF+GB 的 fit-predict |
| 模型性能评估 10% | 混淆矩阵 / P / R / F1 / AUC / 阈值 / class_weight |

## 二、API 速查

| 类 | 关键参数 | 一句话 |
|---|---|---|
| `LogisticRegression(max_iter)` | `class_weight` | 线性基线，有概率输出 |
| `DummyClassifier(strategy)` | — | 哑基线，戳穿准确率幻觉 |
| `KNeighborsClassifier(n_neighbors)` | — | 量纲敏感 |
| `GaussianNB` | — | 缩放免疫 |
| `LinearSVC` / `SVC(probability)` | `gamma` | 前者无概率 |
| `DecisionTreeClassifier(max_depth)` | — | 不限深=背表 |
| `roc_curve` / `roc_auc_score` | — | 阈值扫描 / 排序能力 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier

DATA = Path("data")

dev = pd.read_csv(DATA / "defect_ml.csv")    # 分类：预测「是否危急」

NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
TARGET = "是否危急"
y = dev[TARGET]

# 分层切分：21.3% 的正例比例在训练/测试段保持一致
tr_idx, te_idx = train_test_split(np.arange(len(dev)), test_size=0.2,
                                  random_state=42, stratify=y)
dev_tr, dev_te = dev.iloc[tr_idx], dev.iloc[te_idx]
ys_tr, ys_te = dev_tr[TARGET], dev_te[TARGET]

# 数值列 median 填充 + 独热 + 标准化（预处理细节 ch02/ch03 已讲透）
imp = SimpleImputer(strategy="median").fit(dev_tr[NUM])
oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(dev_tr[CAT])
feat = NUM + [f"{c}_{v}" for c, vals in zip(CAT, oh.categories_) for v in vals]
Xtr = np.hstack([imp.transform(dev_tr[NUM]), oh.transform(dev_tr[CAT])])
Xte = np.hstack([imp.transform(dev_te[NUM]), oh.transform(dev_te[CAT])])
ss = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = ss.transform(Xtr), ss.transform(Xte)

print("sklearn", sklearn.__version__)
print("train", Xtr_s.shape, "正例", int(ys_tr.sum()), "/", len(ys_tr),
      "| test", Xte_s.shape, "正例", int(ys_te.sum()), "/", len(ys_te))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


print("脚手架就绪：probe")

## 题 1：逻辑回归与概率输出（对应讲解 §3.1）

In [ ]:
# TODO(1)：训练逻辑回归，预测测试段并算混淆矩阵
# 提示：LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_s, ys_tr) + predict + confusion_matrix
lr = ____
yp = ____
cm = ____
print("test 预测正例:", int(yp.sum()), "| 实际正例:", int(ys_te.sum()))
print("混淆矩阵 [[TN FP] / [FN TP]]:")
print(cm)

# TODO(2)：取 predict_proba 与 decision_function，验证 decision == logit(p)
# 提示：lr.predict_proba(Xte_s) / lr.decision_function(Xte_s)；np.allclose(dec, np.log(proba[:, 1] / proba[:, 0]))
proba = ____
dec = ____
print("proba 行和:", np.round(proba.sum(axis=1)[:3], 6).tolist(),
      "| classes_:", lr.classes_.tolist())
print("decision range:", round(float(dec.min()), 4), "→", round(float(dec.max()), 4))
print("dec == logit(p) 验证:", bool(np.allclose(dec, np.log(proba[:, 1] / proba[:, 0]))))
top = feat[int(np.argmax(np.abs(lr.coef_[0])))]
print("|coef| 最大:", top, round(float(lr.coef_[0].max()), 4),
      "| intercept:", round(float(lr.intercept_[0]), 4))

assert int(yp.sum()) == 10, "模型只敢报 10 个危急，实际有 26 个"
assert cm.tolist() == [[89, 5], [21, 5]], "FN=21 远大于 FP=5：默认阈值偏向多数类"
assert proba.shape == (120, 2) and lr.classes_.tolist() == [0, 1]
assert float(np.abs(proba.sum(axis=1) - 1).max()) < 1e-9
assert np.allclose(dec, np.log(proba[:, 1] / proba[:, 0])), "decision_function 就是 log-odds"
assert top == "温度" and round(float(lr.coef_[0].max()), 4) == 0.8106
assert round(float(lr.intercept_[0]), 4) == -1.7024

**为什么这样做**：`decision_function == logit(p)` 是逻辑回归的「身份证」——
验证它，你就确认了 proba 不是黑盒输出，阈值移动才有意义。

## 题 2：准确率陷阱（对应讲解 §3.2）

In [ ]:
# TODO(3)：训练全预测负类的哑模型，与逻辑回归的准确率对账
# 提示：DummyClassifier(strategy="most_frequent").fit(Xtr_s, ys_tr)
dummy = ____
yp_d = ____
acc_lr = ____
acc_dm = ____
print(f"逻辑回归 acc={acc_lr:.4f} | 哑模型 acc={acc_dm:.4f}")
print("哑模型 precision:", round(precision_score(ys_te, yp_d, zero_division=0), 4),
      "| recall:", round(recall_score(ys_te, yp_d), 4))
print(classification_report(ys_te, yp, target_names=["一般", "危急"], zero_division=0))

assert round(acc_dm, 4) == 0.7833, "全预测负类 = 94/120"
assert round(acc_lr, 4) == 0.7833, "两者准确率打平——准确率在 21% 正例上失效"
assert precision_score(ys_te, yp_d, zero_division=0) == 0.0
assert recall_score(ys_te, yp_d) == 0.0
assert round(f1_score(ys_te, yp), 4) == 0.2778, "F1 才看得出逻辑回归真的学到了东西"

## 题 3：阈值移动与 Youden J（对应讲解 §3.3）

In [ ]:
# 阈值扫描先给出分数列（脚手架，不用挖空）
p1 = proba[:, 1]
for t in (0.3, 0.5, 0.7):
    yp_t = (p1 >= t).astype(int)
    print(f"t={t}: 预测正例 {int(yp_t.sum()):>2} acc={accuracy_score(ys_te, yp_t):.4f} "
          f"P={precision_score(ys_te, yp_t, zero_division=0):.4f} "
          f"R={recall_score(ys_te, yp_t):.4f}")

# TODO(4)：用 roc_curve 取阈值序列，找 Youden J（TPR-FPR）最大的阈值
# 提示：fpr, tpr, thr = roc_curve(ys_te, p1)；j = int(np.argmax(tpr - fpr))
fpr, tpr, thr = ____
j = ____
print("阈值数:", len(thr), "| 首阈值:", thr[0])
print(f"Youden 最优 t={thr[j]:.4f}: TPR={tpr[j]:.4f} FPR={fpr[j]:.4f}")

assert len(thr) == 39
assert bool(np.isinf(thr[0])), "sklearn 1.9 首阈值是 inf（无样本被预测为正）"
assert round(float(thr[j]), 4) == 0.2455
assert round(float(tpr[j]), 4) == 0.8077 and round(float(fpr[j]), 4) == 0.2660
assert round(recall_score(ys_te, (p1 >= 0.3).astype(int)), 4) == 0.6538

## 题 4：class_weight（对应讲解 §3.4）

In [ ]:
# TODO(5)：训练 class_weight="balanced" 的逻辑回归，对比混淆矩阵
# 提示：LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_b = ____
yp_b = ____
cm_b = ____
print("balanced 混淆矩阵:")
print(cm_b)
print("balanced 预测正例:", int(yp_b.sum()),
      "| recall:", round(recall_score(ys_te, yp_b), 4),
      "| acc:", round(accuracy_score(ys_te, yp_b), 4))

assert cm_b.tolist() == [[63, 31], [5, 21]], "FN 21→5，代价是 FP 5→31"
assert int(yp_b.sum()) == 52, "正例预测数 10 → 52"
assert round(recall_score(ys_te, yp_b), 4) == 0.8077
assert round(accuracy_score(ys_te, yp_b), 4) == 0.7000, "准确率换 recall，看业务要哪个"

## 题 5：KNN 与量纲（对应讲解 §3.5）

In [ ]:
# TODO(6)：训练 KNN k=5 的 scaled 版与 raw 版，对账准确率与 precision
# 提示：KNeighborsClassifier(n_neighbors=5)；scaled 用 Xtr_s/Xte_s，raw 用 Xtr/Xte
knn5 = ____
knn5_raw = ____
acc_s = ____
acc_raw = ____
p_s = ____
p_raw = ____
print(f"KNN k=5 scaled: acc={acc_s:.4f} P={p_s:.4f}")
print(f"KNN k=5 raw   : acc={acc_raw:.4f} P={p_raw:.4f}")

for k in (3, 11):
    knn_k = KNeighborsClassifier(n_neighbors=k).fit(Xtr_s, ys_tr)
    print(f"KNN k={k} scaled: acc={accuracy_score(ys_te, knn_k.predict(Xte_s)):.4f} "
          f"R={recall_score(ys_te, knn_k.predict(Xte_s)):.4f}")

assert round(acc_s, 4) == 0.7500 and round(acc_raw, 4) == 0.8000
assert round(p_raw, 4) == 0.6250 and round(p_s, 4) == 0.3000, "raw 版误报反而更少"
assert round(recall_score(ys_te, knn5.predict(Xte_s)), 4) == 0.1154

## 题 6：朴素贝叶斯缩放免疫（对应讲解 §3.6）

In [ ]:
# TODO(7)：训练 scaled 与 raw 两版高斯朴素贝叶斯，验证预测逐位一致
# 提示：GaussianNB().fit(...) 两版；same = bool((gnb.predict(Xte_s) == gnb_raw.predict(Xte)).all())
gnb = ____
gnb_raw = ____
acc_s = ____
acc_raw = ____
same = ____
print(f"GNB scaled acc={acc_s:.4f} | raw acc={acc_raw:.4f} | 预测逐位一致: {same}")
print("GNB theta_[0] 前 3(类 0 均值):", np.round(gnb.theta_[0][:3], 4).tolist())

assert round(acc_s, 4) == 0.6750 and round(acc_raw, 4) == 0.6750
assert same, "逐列减均值除方差不改变高斯似然的 argmax"
assert round(float(gnb.theta_[0][0]), 4) == -0.0988
assert round(recall_score(ys_te, gnb.predict(Xte_s)), 4) == 0.4231, "NB 敢报正例"

## 题 7：SVM 三兄弟（对应讲解 §3.7）

In [ ]:
# TODO(8)：训练 LinearSVC 与 SVC（scaled/raw 各一），对账三份准确率
# 提示：LinearSVC(max_iter=10000, random_state=42)；SVC(random_state=42)
lsvc = ____
svc = ____
svc_raw = ____
acc_l = ____
acc_v = ____
acc_r = ____
print(f"LinearSVC scaled: acc={acc_l:.4f} P={precision_score(ys_te, lsvc.predict(Xte_s), zero_division=0):.4f}")
print(f"SVC rbf scaled  : acc={acc_v:.4f}")
print(f"SVC rbf raw     : acc={acc_r:.4f} P={precision_score(ys_te, svc_raw.predict(Xte), zero_division=0):.4f} "
      f"R={recall_score(ys_te, svc_raw.predict(Xte)):.4f} ← 全预测负类躺平")

# TODO(9)：验证 LinearSVC 没有 predict_proba；再用 SVC(probability=True) 拿概率
# 提示：probe(getattr, lsvc, "predict_proba")；SVC(probability=True, random_state=42)
st_proba = ____
svc_p = ____
print("LinearSVC.predict_proba →", st_proba)
print("SVC(probability=True) proba shape:", svc_p.predict_proba(Xte_s).shape)

assert round(acc_l, 4) == 0.8000 and round(precision_score(ys_te, lsvc.predict(Xte_s), zero_division=0), 4) == 0.6250
assert round(acc_v, 4) == 0.7583
assert round(acc_r, 4) == 0.7833 and recall_score(ys_te, svc_raw.predict(Xte)) == 0.0
assert st_proba == ("err", "AttributeError"), "LinearSVC 没有 predict_proba"
assert svc_p.predict_proba(Xte_s).shape == (120, 2), "probability=True 走内部 CV 估算"

## 题 8：树与集成（对应讲解 §3.8）

In [ ]:
# TODO(10)：训练全深决策树与 max_depth=5 的剪枝树，对比叶子数与 train/test acc
# 提示：DecisionTreeClassifier(random_state=42)；get_n_leaves()
dt_full = ____
dt5 = ____
leaves_full = ____
leaves5 = ____
train_acc = ____
acc5 = ____
print(f"DT 不限深: {leaves_full} 叶 train acc={train_acc:.4f} test={accuracy_score(ys_te, dt_full.predict(Xte_s)):.4f}")
print(f"DT d=5  : {leaves5} 叶 test acc={acc5:.4f} ← 全场准确率冠军")

# TODO(11)：训练 RF(200 棵) 与 GBDT，读 RF 的特征重要性第一名
# 提示：RandomForestClassifier(n_estimators=200, random_state=42)；pd.Series(rf.feature_importances_, index=feat)
rf = ____
gb = ____
acc_rf = ____
acc_gb = ____
imp_rf = ____
print(f"RF test acc={acc_rf:.4f} | GB test acc={acc_gb:.4f}")
print("RF importance top4:", {k: round(v, 4) for k, v in imp_rf.head(4).items()})

assert leaves_full == 95 and round(train_acc, 4) == 1.0, "背表现场"
assert leaves5 == 18 and round(acc5, 4) == 0.7917
assert round(acc_rf, 4) == 0.7667 and round(acc_gb, 4) == 0.7750
assert round(float(imp_rf.iloc[0]), 4) == 0.2220 and imp_rf.index[0] == "温度"

## 题 9：AUC 复盘（对应讲解 §3.9）

In [ ]:
# TODO(12)：用三份 predict_proba 算 AUC，验证逻辑回归反超集成
# 提示：roc_auc_score(ys_te, proba[:, 1])；rf/gb 用 predict_proba(Xte_s)[:, 1]
auc_lr = ____
auc_rf = ____
auc_gb = ____
print(f"AUC: LR={auc_lr:.4f} > RF={auc_rf:.4f} > GB={auc_gb:.4f}")
print("复盘：准确率上 LR 与哑模型打平、树系反而更高，但 AUC 说 LR 的分数最会排序")

assert round(auc_lr, 4) == 0.7946
assert round(auc_rf, 4) == 0.6991 and round(auc_gb, 4) == 0.6809
assert auc_lr > auc_rf, "不平衡数据上 AUC 比准确率更会讲故事"

## 收官自查

1. 题 1 的 FN=21 说明什么？如果你是运检班组长，敢直接上线这个模型吗？
2. 题 3 的 Youden 阈值 0.2455 换来 TPR 0.8077、FPR 0.2660，这批 FP 的业务代价谁承担？
3. 题 5 与题 6 结论相反（KNN 敏感、GNB 免疫），一句话说清分界线在哪。
4. 题 7 raw 版 SVC 的 P=R=0 与题 2 哑模型的 P=R=0 是同一种躺平吗？
5. 题 9 为什么 LR 的 AUC 能反超树系集成？AUC 到底在量什么？

全部答得上来，进入 ch06（模型评估与交叉验证）。